In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

data_dir = Path("../../datasets").resolve()
data_dir.mkdir(parents=True, exist_ok=True)

print(f"Dataset directory: {data_dir}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root=str(data_dir),
    train=True,
    download=True,
    transform=transform,
)

test_dataset = datasets.MNIST(
    root=str(data_dir),
    train=False,
    download=True,
    transform=transform,
)

batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

classes = tuple(str(i) for i in range(10))
print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Input tensor shape: {train_dataset[0][0].shape}")


## MNIST dataset notes

- **Task**: classify handwritten digits from 0 to 9.
- **Input type**: grayscale image, not RGB. Each image has a single channel.
- **Original image size**: `28 x 28` pixels.
- **Tensor shape after `ToTensor()`**: `(1, 28, 28)`.
- **Pixel range after `ToTensor()`**: floating-point values in `[0, 1]`.
- **Targets**: integer class labels from `0` to `9`.
- **Dataset size**: 60,000 training images and 10,000 test images.
- **Why this matters here**: for standard MNIST classifiers we usually keep the full pipeline simple: tensor conversion, minibatch loading, cross-entropy loss, optimizer, train/test split, and accuracy tracking. The same training loop can later be reused for filtered or enlarged MNIST-like inputs, as long as the model input shape is updated consistently.


## Small classifier

In [ ]:
def count_trainable_parameters(model):
    return sum(param.numel() for param in model.parameters() if param.requires_grad)


def print_model_summary(model, input_shape=(1, 1, 28, 28)):
    print(model)
    print(f"Trainable parameters: {count_trainable_parameters(model):,}")
    with torch.no_grad():
        sample = torch.zeros(input_shape, device=device)
        output = model(sample)
    print(f"Output shape for input {input_shape}: {tuple(output.shape)}")


def evaluate_model(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    with torch.no_grad():
        for inputs, targets in loader:
            inputs = inputs.to(device)
            targets = targets.to(device)

            logits = model(inputs)
            loss = criterion(logits, targets)

            total_loss += loss.item() * inputs.size(0)
            total_correct += (logits.argmax(dim=1) == targets).sum().item()
            total_examples += inputs.size(0)

    return total_loss / total_examples, total_correct / total_examples


def train_model(model, train_loader, test_loader, epochs=8, lr=1e-3, weight_decay=0.0):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        running_correct = 0
        seen_examples = 0
        start_time = time.time()

        for inputs, targets in train_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)

            optimizer.zero_grad()
            logits = model(inputs)
            loss = criterion(logits, targets)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_correct += (logits.argmax(dim=1) == targets).sum().item()
            seen_examples += inputs.size(0)

        train_loss = running_loss / seen_examples
        train_acc = running_correct / seen_examples
        test_loss, test_acc = evaluate_model(model, test_loader, criterion)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["test_loss"].append(test_loss)
        history["test_acc"].append(test_acc)

        elapsed = time.time() - start_time
        print(
            f"Epoch {epoch:02d}/{epochs} | "
            f"train loss {train_loss:.4f} | train acc {train_acc:.4%} | "
            f"test loss {test_loss:.4f} | test acc {test_acc:.4%} | "
            f"time {elapsed:.1f}s"
        )

    return history


def plot_history(history, title):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs, history["train_loss"], marker="o", label="Train")
    axes[0].plot(epochs, history["test_loss"], marker="s", label="Test")
    axes[0].set_title(f"{title}: loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Cross-entropy")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    axes[1].plot(epochs, history["train_acc"], marker="o", label="Train")
    axes[1].plot(epochs, history["test_acc"], marker="s", label="Test")
    axes[1].set_title(f"{title}: accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    plt.tight_layout()
    plt.show()


class SmallMNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


small_model = SmallMNISTCNN().to(device)
print_model_summary(small_model)

print("\nThis is the standard MNIST training procedure we will also reuse later:")
print("1. Build DataLoaders for train/test data.")
print("2. Define model, loss, optimizer.")
print("3. Train over minibatches with gradient updates.")
print("4. Evaluate on held-out test data after each epoch.")
print("5. Plot loss/accuracy history.")
print("For filtered or enlarged images, the same loop still applies; only the transform pipeline and possibly the model input block need to change.")

small_history = train_model(
    small_model,
    train_loader,
    test_loader,
    epochs=8,
    lr=1e-3,
)


In [ ]:
plot_history(small_history, title="Small MNIST CNN")
print(f"Final train accuracy: {small_history['train_acc'][-1]:.4%}")
print(f"Final test accuracy: {small_history['test_acc'][-1]:.4%}")


## Deep precise classifier

In [ ]:
class DeepMNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 10),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


deep_model = DeepMNISTCNN().to(device)
print_model_summary(deep_model)

deep_history = train_model(
    deep_model,
    train_loader,
    test_loader,
    epochs=10,
    lr=8e-4,
    weight_decay=1e-4,
)


In [ ]:
plot_history(deep_history, title="Deep MNIST CNN")
print(f"Final train accuracy: {deep_history['train_acc'][-1]:.4%}")
print(f"Final test accuracy: {deep_history['test_acc'][-1]:.4%}")
print("This deeper classifier is a typical strong MNIST baseline and should usually reach roughly 99% test accuracy when trained cleanly on standard MNIST.")
print("So 92-95% is comfortably achievable; if you later apply harder filtering or image enlargement, this model gives you headroom for accuracy loss.")
